# 15 · Unified Employee Intelligence (Capstone Table)

**Project:** Enterprise HR AI — Capstone Workforce Analytics Table  

> ### ⚠️ DATA INTEGRITY & PROVENANCE HEADER
>
> **This table combines: (1) a validated ML risk model (see model_card.md), (2) real engagement survey data covering 49.7% of employees (Step 13 -- do not generalize to the rest), (3) O*NET role mappings with confidence levels ranging from very_low to medium -- none are exact matches (see data_relationships.md Open Issue #1), and (4) SYNTHETIC skill-gap and recommendation data (Steps 15-18) that has NOT been validated against real employee skill records. Sections 3 and 4 are illustrative of MVP capability, not production-ready HR guidance.**

---

---
## Step 1 · Anchor Table: Load Full 1,470-Employee Workforce

Anchor table: `data/processed/employee_attrition_processed.csv`.  
Rule: All 1,470 employees are preserved across all subsequent left joins. Never subset.

In [1]:
import pandas as pd
import numpy as np
import joblib
import json
import os
import warnings
warnings.filterwarnings('ignore')

PROC   = os.path.join('..', 'data', 'processed')
MODELS = os.path.join('..', 'models')

# Load anchor table
df_anchor = pd.read_csv(os.path.join(PROC, 'employee_attrition_processed.csv'))
print(f'Anchor table loaded: {len(df_anchor):,} employees')
assert len(df_anchor) == 1470, f'Expected 1,470 anchor rows, got {len(df_anchor)}'

# Base unified table starting from anchor
unified_df = df_anchor[['EmployeeNumber', 'Department', 'JobRole']].copy()
print(f'Initial anchor row count: {len(unified_df):,}')

Anchor table loaded: 1,470 employees
Initial anchor row count: 1,470


---
## Step 2 · Join 1: Attrition Risk Model Scoring

Load production model `attrition_pipeline.joblib` and `model_config.json`.  
Score all 1,470 employees using `features_scaled.csv`. Threshold = 0.40.

In [2]:
# Load production model & config
model = joblib.load(os.path.join(MODELS, 'attrition_pipeline.joblib'))
with open(os.path.join(MODELS, 'model_config.json'), 'r') as f:
    model_cfg = json.load(f)

threshold = model_cfg['threshold']
print(f'Model algorithm : {model_cfg["model"]}')
print(f'Decision threshold: {threshold}')

# Load scaled features matching model input
fs = pd.read_csv(os.path.join(PROC, 'features_scaled.csv'))
X = fs.drop(columns=['Attrition'])
probs = model.predict_proba(X)[:, 1]

df_risk = pd.DataFrame({
    'EmployeeNumber': df_anchor['EmployeeNumber'],
    'RiskScore': probs.round(4),
    'RiskLevel': ['HIGH' if p >= threshold else 'LOW' for p in probs]
})

# Left Join 1
unified_df = unified_df.merge(df_risk, on='EmployeeNumber', how='left')
print(f'Row count after Join 1 (Risk Scoring): {len(unified_df):,}')
assert len(unified_df) == 1470, f'Join 1 altered row count! {len(unified_df)}'

Model algorithm : logistic_regression_balanced
Decision threshold: 0.4
Row count after Join 1 (Risk Scoring): 1,470


---
## Step 3 · Join 2: Engagement Survey Overlay

Load `data/processed/employee_intelligence_partial.csv` to overlay `Engagement Score`, `Satisfaction Score`, and `Work-Life Balance Score`.  
Retain null values for the 739 unmapped employees (do not impute).

In [3]:
df_eng = pd.read_csv(os.path.join(PROC, 'employee_intelligence_partial.csv'))
eng_subset = df_eng[['EmployeeNumber', 'Engagement Score', 'Satisfaction Score', 'Work-Life Balance Score']].copy()
eng_subset.rename(columns={
    'Engagement Score': 'EngagementScore',
    'Satisfaction Score': 'SatisfactionScore',
    'Work-Life Balance Score': 'WorkLifeBalanceScore'
}, inplace=True)

# Left Join 2
unified_df = unified_df.merge(eng_subset, on='EmployeeNumber', how='left')
print(f'Row count after Join 2 (Engagement Survey): {len(unified_df):,}')
assert len(unified_df) == 1470, f'Join 2 altered row count! {len(unified_df)}'
print(f'  Non-null EngagementScore: {unified_df["EngagementScore"].notnull().sum():,}')
print(f'  Null EngagementScore    : {unified_df["EngagementScore"].isnull().sum():,}')

Row count after Join 2 (Engagement Survey): 1,470
  Non-null EngagementScore: 731
  Null EngagementScore    : 739


---
## Step 4 · Join 3: O*NET Role Intelligence Profiles

Load `role_skill_profiles.csv` to attach mapped `ONET_Title` and `ONET_Confidence` based on `JobRole`.

In [4]:
df_rsp = pd.read_csv(os.path.join(PROC, 'role_skill_profiles.csv'))
onet_subset = df_rsp[['ibm_job_role', 'onet_title', 'match_confidence']].copy()
onet_subset.rename(columns={
    'ibm_job_role': 'JobRole',
    'onet_title': 'ONET_Title',
    'match_confidence': 'ONET_Confidence'
}, inplace=True)

# Left Join 3
unified_df = unified_df.merge(onet_subset, on='JobRole', how='left')
print(f'Row count after Join 3 (O*NET Profiles): {len(unified_df):,}')
assert len(unified_df) == 1470, f'Join 3 altered row count! {len(unified_df)}'

Row count after Join 3 (O*NET Profiles): 1,470


---
## Step 5 · Join 4: Individual Skill Gaps & Severity

Load `data/processed/employee_skill_gaps.csv` for `SkillGapCount` and `SkillGapSeverity`.  
Manager employees (102 records) explicitly assigned `'N/A - Manager'` for severity and `NaN` for count.

In [5]:
df_gaps = pd.read_csv(os.path.join(PROC, 'employee_skill_gaps.csv'), comment='#')
gaps_subset = df_gaps[['EmployeeNumber', 'gap_count', 'severity']].copy()
gaps_subset.rename(columns={
    'gap_count': 'SkillGapCount',
    'severity': 'SkillGapSeverity'
}, inplace=True)

# Left Join 4
unified_df = unified_df.merge(gaps_subset, on='EmployeeNumber', how='left')

# Handle Manager exclusion explicitly
unified_df.loc[unified_df['JobRole'] == 'Manager', 'SkillGapSeverity'] = 'N/A - Manager'

print(f'Row count after Join 4 (Skill Gaps): {len(unified_df):,}')
assert len(unified_df) == 1470, f'Join 4 altered row count! {len(unified_df)}'

Row count after Join 4 (Skill Gaps): 1,470


---
## Step 6 · Join 5: Top 3 Training Recommendations

Load `data/processed/employee_recommendations.csv` for `Top3Recommendations`.  
Manager role assigned `'N/A - Manager (use Department-level analysis)'`.

In [6]:
df_recs = pd.read_csv(os.path.join(PROC, 'employee_recommendations.csv'), comment='#')
recs_subset = df_recs[['EmployeeNumber', 'top_3_recommendations']].copy()
recs_subset.rename(columns={
    'top_3_recommendations': 'Top3Recommendations'
}, inplace=True)

# Left Join 5
unified_df = unified_df.merge(recs_subset, on='EmployeeNumber', how='left')

# Handle Manager exclusion explicitly
unified_df.loc[unified_df['JobRole'] == 'Manager', 'Top3Recommendations'] = 'N/A - Manager (use Department-level analysis)'

print(f'Row count after Join 5 (Recommendations): {len(unified_df):,}')
assert len(unified_df) == 1470, f'Join 5 altered row count! {len(unified_df)}'

Row count after Join 5 (Recommendations): 1,470


---
## Step 7 · Finalize Columns & Complete Integrity Verification

Ordering exactly as requested:
`EmployeeNumber, Department, JobRole, RiskScore, RiskLevel, EngagementScore, SatisfactionScore, WorkLifeBalanceScore, ONET_Title, ONET_Confidence, SkillGapCount, SkillGapSeverity, Top3Recommendations`

In [7]:
FINAL_COLUMNS = [
    'EmployeeNumber', 'Department', 'JobRole', 'RiskScore', 'RiskLevel',
    'EngagementScore', 'SatisfactionScore', 'WorkLifeBalanceScore',
    'ONET_Title', 'ONET_Confidence', 'SkillGapCount', 'SkillGapSeverity',
    'Top3Recommendations'
]

df_final = unified_df[FINAL_COLUMNS].copy()

print('=== FINAL DATA INTEGRITY CONFIRMATION ===')
print(f'Total workforce rows : {len(df_final):,} (Confirmed: exactly 1,470)')
print(f'Total columns        : {len(df_final.columns)}')
print(f'Column names         : {list(df_final.columns)}')
assert len(df_final) == 1470, 'Final row count must be 1,470!'

print('\nFirst 3 Records:')
print(df_final.head(3).to_string(index=False))

print('\nManager Records (Validation of N/A Exclusion):')
print(df_final[df_final['JobRole'] == 'Manager'][['EmployeeNumber', 'JobRole', 'ONET_Title', 'ONET_Confidence', 'SkillGapSeverity', 'Top3Recommendations']].head(2).to_string(index=False))

=== FINAL DATA INTEGRITY CONFIRMATION ===
Total workforce rows : 1,470 (Confirmed: exactly 1,470)
Total columns        : 13
Column names         : ['EmployeeNumber', 'Department', 'JobRole', 'RiskScore', 'RiskLevel', 'EngagementScore', 'SatisfactionScore', 'WorkLifeBalanceScore', 'ONET_Title', 'ONET_Confidence', 'SkillGapCount', 'SkillGapSeverity', 'Top3Recommendations']

First 3 Records:
 EmployeeNumber             Department               JobRole  RiskScore RiskLevel  EngagementScore  SatisfactionScore  WorkLifeBalanceScore                                   ONET_Title ONET_Confidence  SkillGapCount SkillGapSeverity                                                                                                                                                                                                       Top3Recommendations
              1                  Sales       Sales Executive     0.8976      HIGH              NaN                NaN                   NaN                  

---
## Step 8 · Save Unified Dataset (`employee_intelligence.csv`)

Saving the capstone intelligence table to `data/processed/employee_intelligence.csv`.
The provenance warning header is permanently embedded in line 1 of the file.

In [8]:
out_path = os.path.join(PROC, 'employee_intelligence.csv')

header_comment = (
    '# This table combines: (1) a validated ML risk model (see model_card.md), '
    '(2) real engagement survey data covering 49.7% of employees (Step 13 -- do not generalize to the rest), '
    '(3) O*NET role mappings with confidence levels ranging from very_low to medium -- none are exact matches '
    '(see data_relationships.md Open Issue #1), and (4) SYNTHETIC skill-gap and recommendation data (Steps 15-18) '
    'that has NOT been validated against real employee skill records. Sections 3 and 4 are illustrative '
    'of MVP capability, not production-ready HR guidance.\n'
)

with open(out_path, 'w', encoding='utf-8') as f:
    f.write(header_comment)
    df_final.to_csv(f, index=False)

file_size = os.path.getsize(out_path)
print(f'Saved capstone table to : {out_path}')
print(f'File size               : {file_size:,} bytes')
print(f'Total rows              : {len(df_final):,}')

# Round-trip reload verification
df_reload = pd.read_csv(out_path, comment='#')
assert len(df_reload) == 1470, 'Row count mismatch on reload!'
assert list(df_reload.columns) == FINAL_COLUMNS, 'Columns mismatch on reload!'
print('CONFIRMED: Round-trip verification passed cleanly.')

Saved capstone table to : ..\data\processed\employee_intelligence.csv
File size               : 384,229 bytes
Total rows              : 1,470
CONFIRMED: Round-trip verification passed cleanly.


---
## Step 9 · Summary Statistics & Completeness Breakdown

In [9]:
print('=== FINAL SUMMARY STATISTICS (N=1,470) ===\n')

# 1. RiskLevel Distribution
risk_dist = df_final['RiskLevel'].value_counts()
risk_pcts = (df_final['RiskLevel'].value_counts(normalize=True) * 100).round(2)
print('1. RiskLevel Distribution:')
for level, count in risk_dist.items():
    print(f'   {level:<5}: {count:>4} employees ({risk_pcts[level]:>5.2f}%)')

# 2. SkillGapSeverity Distribution
gap_dist = df_final['SkillGapSeverity'].value_counts()
gap_pcts = (df_final['SkillGapSeverity'].value_counts(normalize=True) * 100).round(2)
print('\n2. SkillGapSeverity Distribution:')
for sev, count in gap_dist.items():
    print(f'   {sev:<15}: {count:>4} employees ({gap_pcts[sev]:>5.2f}%)')

# 3. Data Completeness Across All 5 Sources
complete_mask = (
    df_final['RiskScore'].notnull() &
    df_final['EngagementScore'].notnull() &
    df_final['ONET_Title'].notnull() &
    df_final['SkillGapCount'].notnull() &
    (df_final['JobRole'] != 'Manager')
)
n_complete = complete_mask.sum()
n_partial = len(df_final) - n_complete

print('\n3. Workforce Data Completeness Across All 5 Sources:')
print(f'   Complete Data (all 5 sources present, non-manager) : {n_complete:>4} ({n_complete/1470*100:.2f}%)')
print(f'   Partial Data (missing survey and/or Manager role)  : {n_partial:>4} ({n_partial/1470*100:.2f}%)')
print(f'     - Missing engagement survey (known 49.7% sample) : {df_final["EngagementScore"].isnull().sum():>4} employees')
print(f'     - Manager role (O*NET skill gap exclusion)       : {(df_final["JobRole"] == "Manager").sum():>4} employees')
print(f'     - Overlap (both missing survey & Manager)        : {((df_final["EngagementScore"].isnull()) & (df_final["JobRole"] == "Manager")).sum():>4} employees')

=== FINAL SUMMARY STATISTICS (N=1,470) ===

1. RiskLevel Distribution:
   LOW  :  885 employees (60.20%)
   HIGH :  585 employees (39.80%)

2. SkillGapSeverity Distribution:
   LOW            :  870 employees (59.18%)
   MEDIUM         :  427 employees (29.05%)
   N/A - Manager  :  102 employees ( 6.94%)
   HIGH           :   71 employees ( 4.83%)

3. Workforce Data Completeness Across All 5 Sources:
   Complete Data (all 5 sources present, non-manager) :  682 (46.39%)
   Partial Data (missing survey and/or Manager role)  :  788 (53.61%)
     - Missing engagement survey (known 49.7% sample) :  739 employees
     - Manager role (O*NET skill gap exclusion)       :  102 employees
     - Overlap (both missing survey & Manager)        :   53 employees
